In [45]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [46]:
import numpy as np
from scipy.optimize import minimize

from src.data_loader import get_prices
from src.features import calculate_returns
from src.portfolio import (
    portfolio_return,
    portolio_volatility,
    sharpe_ratio,
    negative_sharpe
)

In [47]:
tickers = ["SPY", "QQQ", "TSLA", "PLTR"]

prices = get_prices(tickers)
returns = calculate_returns(prices)

mean_returns = returns.mean()
cov_matrix = returns.cov()

[*********************100%***********************]  4 of 4 completed


In [48]:
num_assets = len(tickers)
init_guess = np.array([1/num_assets] * num_assets)
constraints = (
    {"type": "eq", "fun": lambda w: np.sum(w) - 1}
)
bounds = tuple((0, 1) for _ in range(num_assets))

In [49]:
result = minimize(
    negative_sharpe,
    init_guess,
    args=(mean_returns, cov_matrix),
    method="SLSQP",
    bounds = bounds,
    constraints=constraints
)

In [50]:
optimal_weights = result.x

print("Optimal Weights:")
for ticker, weight in zip(tickers, optimal_weights):
    print(f"{ticker}: {weight:.4f}")

Optimal Weights:
SPY: 0.0956
QQQ: 0.0000
TSLA: 0.8520
PLTR: 0.0524


In [51]:
opt_return = portfolio_return(optimal_weights, mean_returns)
opt_vol = portolio_volatility(optimal_weights, cov_matrix)
opt_sharpe = sharpe_ratio(optimal_weights, mean_returns, cov_matrix)

print("\nOptimal Portfolio Metrics:")
print("Return:", opt_return)
print("Volatility", opt_vol)
print("Sharpe:", opt_sharpe)


Optimal Portfolio Metrics:
Return: 0.0007091174619063782
Volatility 0.013291594638679146
Sharpe: 0.0533508191592613
